# Detecting coordinated players in poker — an end-to-end baseline

This notebook goes from the raw competition files to a validated `submission.csv`:

1. load and sanity-check the eight release files
2. a self-contained 7-card hand evaluator (verified in-notebook)
3. one pass over the action log to build **pair-hand** rows (partner-versus-field responses, retrospective strength, chip flow)
4. pair-level aggregation
5. **risk model** (pair ranking) with whole-pool cross-validation
6. **behavior head** (which family)
7. **evidence ranker** (which five hands), routed by the predicted family
8. the official metric, computed locally out-of-fold
9. evaluation inference and a validated submission file

**Scope, stated honestly.** This is a compact reimplementation of the *approach* used for a submission that
scored 0.82412 on the public leaderboard. It is **not** that file: the original used a 700-field pair model,
930-field evidence rankers with a decision-equity block, and a multi-hour build. Expect this notebook to
score meaningfully lower. Everything here is reproducible from the released data alone.

**Rules this notebook keeps.** Predictions come from gameplay only — never from identifiers, row order, file
order, label status, or the evaluation-inclusion filter. Unlabelled development pairs are never treated as
confirmed negatives: they are simply not used as supervision. Validation splits whole pools, so no player
appears on both sides. The actor's decision-time information is kept separate from the investigator's
retrospective view (hole cards and final boards are used only for post-hoc features, which is what the
operator-side task intends).

**Runtime.** Roughly 1.5–3 hours on a 4-core Kaggle CPU session, dominated by the evaluation pass over
800,000 hands. Set `POOL_LIMIT` to a small number for a fast smoke run.


In [ ]:
import gc, json, math, os, sys, time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

SEED = 20260918
N_FOLDS = 5
POOL_LIMIT = None      # e.g. 6 for a smoke run; None = all 400 pools
WORKERS = max(1, (os.cpu_count() or 2) - 1)

CANDIDATES = [Path('/kaggle/input/detect-suspicious-value-transfers-in-poker'), Path('data'), Path('../data')]
DATA_DIR = next((p for p in CANDIDATES if (p / 'hands.parquet').exists()), None)
if DATA_DIR is None:
    raise SystemExit('Could not find the competition files; set DATA_DIR manually.')
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('data:', DATA_DIR, '| workers:', WORKERS)


## 1. Load and check

`shared_hands` in `evaluation_pairs.csv` is an exposure count, not a label. The three disclosed families are
`directed_transfer`, `soft_play` and `coordinated_isolation`; a fourth mechanism exists in the data but has
no public positive labels, so nothing here can be trained on it.


In [ ]:
labels = pd.read_csv(DATA_DIR / 'development_labels.csv')
evidence = pd.read_csv(DATA_DIR / 'development_evidence.csv')
eval_pairs = pd.read_csv(DATA_DIR / 'evaluation_pairs.csv')
template = pd.read_csv(DATA_DIR / 'sample_submission.csv')
hands = pq.read_table(DATA_DIR / 'hands.parquet',
                      columns=['hand_id', 'table_id', 'phase', 'big_blind', 'board_cards']).to_pandas()

print('labels', labels.shape, '| positives', int(labels.label.sum()), '| evidence rows', len(evidence))
print('evaluation pairs', len(eval_pairs), '| template', len(template))
print(labels.behavior_family.value_counts().to_dict())
print(hands.phase.value_counts().to_dict(), '| pools', hands.table_id.nunique())

pools = sorted(hands.table_id.unique())
if POOL_LIMIT:
    pools = pools[:POOL_LIMIT]
    keep = set(pools)
    hands = hands[hands.table_id.isin(keep)]
    print('SMOKE RUN on', len(pools), 'pools')

pair_key = lambda a, b: (a, b) if a < b else (b, a)
dev_pairs = labels[['pair_id', 'player_1', 'player_2', 'label', 'behavior_family']].copy()
truth_evidence = evidence.sort_values(['pair_id', 'evidence_rank']).groupby('pair_id').hand_id.apply(list).to_dict()


## 2. A 7-card evaluator

Used only for retrospective features: at the final board, who actually held the better hand. Ranks are
`rank*4 + suit`; a hand value is `category * 15**5 + kickers`, higher is better. The next cell checks the
ordering on known examples.


In [ ]:
RANKS, SUITS = '23456789TJQKA', 'cdhs'
CARD = {r + s: RANKS.index(r) * 4 + SUITS.index(s) for r in RANKS for s in SUITS}

def straight_lookup():
    table = np.full(1 << 13, -1, np.int8)
    for mask in range(1 << 13):
        if mask & 0b1000000001111 == 0b1000000001111:
            table[mask] = 3                      # wheel: A-2-3-4-5
        for top in range(4, 13):
            window = sum(1 << r for r in range(top - 4, top + 1))
            if mask & window == window:
                table[mask] = top
    return table

STRAIGHT = straight_lookup()

def hand_values(cards):
    """cards: (n, 5..7) int array of card codes -> (n,) comparable strength values."""
    cards = np.asarray(cards)
    n = len(cards)
    ranks, suits = cards >> 2, cards & 3
    counts = np.zeros((n, 13), np.int8)
    np.add.at(counts, (np.repeat(np.arange(n), cards.shape[1]), ranks.ravel()), 1)
    suit_counts = np.zeros((n, 4), np.int8)
    np.add.at(suit_counts, (np.repeat(np.arange(n), cards.shape[1]), suits.ravel()), 1)
    present = (counts > 0).astype(np.int64)
    mask = present @ (1 << np.arange(13))
    straight = STRAIGHT[mask]
    flush_suit = suit_counts.argmax(1)
    has_flush = suit_counts.max(1) >= 5
    in_flush = (suits == flush_suit[:, None]) & has_flush[:, None]
    fmask = np.zeros((n, 13), np.int64)
    fmask[np.repeat(np.arange(n), cards.shape[1])[in_flush.ravel()], ranks.ravel()[in_flush.ravel()]] = 1
    flush_bits = fmask @ (1 << np.arange(13))
    straight_flush = np.where(has_flush, STRAIGHT[flush_bits], -1)

    def nth(matrix, k):
        """rank index of the k-th highest True column, or -1"""
        desc = matrix[:, ::-1]
        hit = desc & (np.cumsum(desc, 1) == k)
        return np.where(hit.any(1), 12 - hit.argmax(1), -1)

    quad, trip1 = nth(counts == 4, 1), nth(counts == 3, 1)
    trip2, pair1, pair2 = nth(counts == 3, 2), nth(counts == 2, 1), nth(counts == 2, 2)
    singles, anyc = counts == 1, counts > 0
    value = np.zeros(n, np.int64)
    done = np.zeros(n, bool)

    def encode(cat, *slots):
        out = np.full(n, cat, np.int64)
        for i in range(5):
            out = out * 15 + (np.maximum(slots[i], -1) + 1 if i < len(slots) else 0)
        return out

    def assign(cond, encoded):
        nonlocal value, done
        use = cond & ~done
        value = np.where(use, encoded, value)
        done |= use

    assign(straight_flush >= 0, encode(8, straight_flush))
    assign(quad >= 0, encode(7, quad, nth(anyc & (counts != 4), 1)))
    full_pair = np.maximum(trip2, pair1)
    assign((trip1 >= 0) & (full_pair >= 0), encode(6, trip1, full_pair))
    assign(has_flush, encode(5, *[nth(fmask.astype(bool), k) for k in range(1, 6)]))
    assign(straight >= 0, encode(4, straight))
    assign(trip1 >= 0, encode(3, trip1, nth(singles, 1), nth(singles, 2)))
    not_pair = anyc & (np.arange(13) != pair1[:, None]) & (np.arange(13) != pair2[:, None])
    assign(pair2 >= 0, encode(2, pair1, pair2, nth(not_pair, 1)))
    assign(pair1 >= 0, encode(1, pair1, nth(singles, 1), nth(singles, 2), nth(singles, 3)))
    assign(~done, encode(0, *[nth(anyc, k) for k in range(1, 6)]))
    return value


In [ ]:
def parse(hand_string):
    return [CARD[c] for c in hand_string.split()]

examples = {
    'straight flush': '9h 8h 7h 6h 5h 2c 2d',
    'quads':          'As Ah Ad Ac 9h 2c 3d',
    'full house':     'Ks Kh Kd 9c 9h 2s 3d',
    'flush':          'Ah Jh 9h 6h 3h 2c 4d',
    'straight':       '9h 8c 7d 6s 5h 2c 3d',
    'trips':          '7h 7c 7d Ks 9h 2c 3d',
    'two pair':       'Qs Qh 8d 8c 4h 2c 3d',
    'one pair':       'Ts Th 8d 6c 4h 2c 3d',
    'high card':      'As Jh 9d 6c 4h 2c 3s',
}
vals = hand_values(np.array([parse(v) for v in examples.values()]))
order = [k for _, k in sorted(zip(vals, examples), reverse=True)]
print(' > '.join(order))
assert order == list(examples), 'evaluator ordering is wrong'
print('evaluator OK')


## 3. One pass over the action log

For every hand we replay the ordered actions, tracking the current street, who set the price being faced,
and who is still live. From that we emit one row per **pair of interest per shared hand** with:

* **partner-versus-field responses** — how each member folds / calls / raises when *the partner* set the
  price, against the same rates when an *outsider* set it. Coordination is a pair-specific deviation, so the
  contrast matters more than the raw rate.
* **joint entry** — both voluntarily entering the same pot, and both entering with weak holdings.
* **retrospective strength** — at the final board, who held the better hand, and whether the player who lost
  money held the better one.
* **chip flow** — contributions, net results, and an opposing-outcome proxy.

Pairs of interest are the 1,860 labelled development pairs (supervision) and the 112,540 evaluation pairs
(inference). Evaluation hands are processed pool by pool so memory stays bounded.


In [ ]:
AGGRESSIVE = {'bet', 'raise'}
PRE, POST = 0, 1

def chen(c1, c2):
    """Chen-style preflop strength; higher is stronger. Used only as a coarse 'weak holding' flag."""
    r1, r2 = c1 >> 2, c2 >> 2
    hi, lo = max(r1, r2), min(r1, r2)
    base = {12: 10.0, 11: 8.0, 10: 7.0, 9: 6.0}.get(hi, (hi + 2) / 2.0)
    score = base * (2 if r1 == r2 else 1)
    if r1 == r2:
        score = max(score, 5.0)
    if (c1 & 3) == (c2 & 3):
        score += 2
    gap = hi - lo - 1
    score -= {0: 0, 1: 1, 2: 2, 3: 4}.get(gap, 5)
    if gap <= 1 and hi < 10 and r1 != r2:
        score += 1
    return score

def classify(action, amount, to_call):
    if action == 'fold':
        return 0
    if action in AGGRESSIVE or (action == 'all_in' and amount > to_call):
        return 2
    return 1

FEATURES = ['both_vpip', 'both_weak_entry', 'both_showdown', 'pot_bb', 'net_gap_bb', 'opposing_proxy_bb',
            'fold_vs_partner', 'call_vs_partner', 'raise_vs_partner', 'n_partner_priced',
            'fold_vs_field', 'call_vs_field', 'raise_vs_field', 'n_field_priced',
            'hu_postflop_checks', 'hu_postflop_bets', 'partner_aggr_while_live',
            'loser_had_better_hand', 'winner_not_best', 'strength_gap', 'weak_entry_after_partner']

def pool_pair_hands(task):
    pool, phase, wanted = task              # wanted: dict {(p_low, p_high): pair_id}
    hand_tbl = ds.dataset(DATA_DIR / 'hands.parquet').to_table(
        filter=(ds.field('table_id') == pool) & (ds.field('phase') == phase),
        columns=['hand_id', 'big_blind', 'board_cards']).to_pandas()
    if not len(hand_tbl):
        return pd.DataFrame(columns=['pair_id', 'hand_id', *FEATURES])
    ids = hand_tbl.hand_id.tolist()
    seats = ds.dataset(DATA_DIR / 'seats.parquet').to_table(
        filter=ds.field('hand_id').isin(ids),
        columns=['hand_id', 'player_id', 'seat_no', 'hole_card_1', 'hole_card_2',
                 'total_contribution', 'net_chips', 'folded', 'went_to_showdown']).to_pandas()
    actions = ds.dataset(DATA_DIR / 'actions.parquet').to_table(
        filter=ds.field('hand_id').isin(ids),
        columns=['hand_id', 'action_no', 'street', 'player_id', 'action', 'amount', 'to_call']).to_pandas()
    seats = seats.sort_values(['hand_id', 'seat_no'])
    actions = actions.sort_values(['hand_id', 'action_no'])
    seat_groups = {h: g for h, g in seats.groupby('hand_id', sort=False)}
    act_groups = {h: g for h, g in actions.groupby('hand_id', sort=False)}

    rows = []
    for hand in hand_tbl.itertuples(index=False):
        s = seat_groups.get(hand.hand_id)
        a = act_groups.get(hand.hand_id)
        if s is None or a is None:
            continue
        players = s.player_id.to_numpy()
        pairs = [(i, j, wanted[key]) for i in range(len(players)) for j in range(i + 1, len(players))
                 if (key := (players[i], players[j]) if players[i] < players[j] else (players[j], players[i])) in wanted]
        if not pairs:
            continue
        bb = float(hand.big_blind)
        holes = np.array([[CARD[c1], CARD[c2]] for c1, c2 in zip(s.hole_card_1, s.hole_card_2)])
        chens = np.array([chen(h[0], h[1]) for h in holes])
        contrib = s.total_contribution.to_numpy() / bb
        net = s.net_chips.to_numpy() / bb
        showdown = s.went_to_showdown.to_numpy()
        n = len(players)
        index = {p: k for k, p in enumerate(players)}

        board = [CARD[c] for c in (hand.board_cards or '').split()]
        strength = np.full(n, np.nan)
        if len(board) == 5:
            strength = hand_values(np.concatenate([holes, np.tile(board, (n, 1))], axis=1)).astype(float)

        # replay
        resp = np.zeros((n, n, 3))      # actor, price setter, class
        field = np.zeros((n, 3))        # actor, class (price set by anyone)
        vpip = np.zeros(n, bool)
        entered_order = []
        aggr_while = np.zeros((n, n))    # actor bets/raises while the other is live
        hu_checks = np.zeros((n, n))
        hu_bets = np.zeros((n, n))
        live = set(range(n))
        street, setter = None, -1
        for act in a.itertuples(index=False):
            k = index.get(act.player_id)
            if k is None:
                continue
            if act.street != street:
                street, setter = act.street, -1
            post = act.street != 'preflop'
            cls = classify(act.action, act.amount, act.to_call)
            priced = act.to_call > 0
            if priced:
                field[k, cls] += 1
                if setter >= 0:
                    resp[k, setter, cls] += 1
            if not post and cls > 0 and act.amount > 0 and not vpip[k]:
                vpip[k] = True
                entered_order.append(k)
            if cls == 2:
                for other in live:
                    if other != k:
                        aggr_while[k, other] += 1
            if post and not priced and len(live) == 2:
                other = next(iter(live - {k}), None)
                if other is not None:
                    (hu_checks if cls == 1 else hu_bets)[k, other] += 1
            if cls == 0:
                live.discard(k)
            elif cls == 2:
                setter = k

        for i, j, pid in pairs:
            both_in = bool(vpip[i] and vpip[j])
            weak_both = bool(both_in and chens[i] < 8 and chens[j] < 8)
            second_weak = 0.0
            if both_in and len(entered_order) >= 2:
                order = [k for k in entered_order if k in (i, j)]
                if len(order) == 2 and chens[order[1]] < 8:
                    second_weak = 1.0
            partner = resp[i, j] + resp[j, i]
            outsiders = (field[i] + field[j]) - partner
            loser, winner = (i, j) if net[i] < net[j] else (j, i)
            strength_gap = float(strength[i] - strength[j]) if np.isfinite(strength[i]) and np.isfinite(strength[j]) else np.nan
            loser_better = float(strength[loser] > strength[winner]) if np.isfinite(strength_gap) else np.nan
            winner_not_best = float(np.isfinite(strength).any() and strength[winner] < np.nanmax(strength)) if np.isfinite(strength_gap) else np.nan
            rows.append((pid, hand.hand_id, float(both_in), float(weak_both),
                         float(showdown[i] and showdown[j]), float(contrib.sum()), float(abs(net[i] - net[j])),
                         float(min(max(-net[i], 0), max(net[j], 0)) + min(max(-net[j], 0), max(net[i], 0))),
                         partner[0], partner[1], partner[2], float(partner.sum()),
                         outsiders[0], outsiders[1], outsiders[2], float(outsiders.sum()),
                         float(hu_checks[i, j] + hu_checks[j, i]), float(hu_bets[i, j] + hu_bets[j, i]),
                         float(aggr_while[i, j] + aggr_while[j, i]),
                         loser_better, winner_not_best, strength_gap, second_weak))
    return pd.DataFrame(rows, columns=['pair_id', 'hand_id', *FEATURES])


## 4. Pair-level aggregation

Rates over the pair's shared hands, plus the partner-versus-field contrasts that make the signal
pair-specific rather than a property of one loose player.


In [ ]:
RATE_COLS = ['both_vpip', 'both_weak_entry', 'both_showdown', 'loser_had_better_hand', 'winner_not_best',
             'weak_entry_after_partner']
MEAN_COLS = ['pot_bb', 'net_gap_bb', 'opposing_proxy_bb', 'strength_gap']

def aggregate_pairs(frame):
    if not len(frame):
        return pd.DataFrame()
    g = frame.groupby('pair_id', sort=False)
    out = g[RATE_COLS + MEAN_COLS].mean()
    out['shared_hands'] = g.size()
    out['max_opposing_proxy'] = g.opposing_proxy_bb.max()
    out['p90_opposing_proxy'] = g.opposing_proxy_bb.quantile(.9)
    sums = g[['fold_vs_partner', 'call_vs_partner', 'raise_vs_partner', 'n_partner_priced',
              'fold_vs_field', 'call_vs_field', 'raise_vs_field', 'n_field_priced',
              'hu_postflop_checks', 'hu_postflop_bets', 'partner_aggr_while_live']].sum()
    for kind in ('fold', 'call', 'raise'):
        p = sums[f'{kind}_vs_partner'] / sums.n_partner_priced.clip(lower=1)
        f = sums[f'{kind}_vs_field'] / sums.n_field_priced.clip(lower=1)
        out[f'{kind}_rate_partner'] = p
        out[f'{kind}_rate_field'] = f
        out[f'{kind}_contrast'] = p - f            # the pair-specific deviation
    out['partner_priced_n'] = sums.n_partner_priced
    out['hu_check_rate'] = sums.hu_postflop_checks / (sums.hu_postflop_checks + sums.hu_postflop_bets).clip(lower=1)
    out['aggr_while_live_per_hand'] = sums.partner_aggr_while_live / out.shared_hands
    return out.reset_index()

PAIR_FEATURES = None   # set after the first aggregation


## 5. Development pass and the risk model

Supervision is the 1,860 trusted labels only: 372 confirmed targets and 1,488 confirmed non-targets. The
other ~172,000 development relationships are unknown and are left out of the loss entirely — an absent
label is not a negative.

Cross-validation splits **whole pools**, so no player appears in both train and validation.


In [ ]:
t0 = time.time()
wanted_dev = {pair_key(r.player_1, r.player_2): r.pair_id for r in dev_pairs.itertuples(index=False)}
dev_tasks = [(p, 'development', wanted_dev) for p in pools]
frames = []
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for i, part in enumerate(ex.map(pool_pair_hands, dev_tasks), 1):
        if len(part):
            frames.append(part)
        if i % 50 == 0:
            print(f'development {i}/{len(dev_tasks)} pools, {time.time()-t0:.0f}s', flush=True)
dev_hand_rows = pd.concat(frames, ignore_index=True)
del frames; gc.collect()
print('development pair-hand rows:', len(dev_hand_rows), f'{time.time()-t0:.0f}s')

dev_features = aggregate_pairs(dev_hand_rows).merge(dev_pairs, on='pair_id', how='inner')
seat_pool = pq.read_table(DATA_DIR / 'seats.parquet', columns=['hand_id', 'player_id']).to_pandas()
seat_pool = seat_pool.merge(hands[['hand_id', 'table_id']], on='hand_id').drop_duplicates(['player_id'])
player_pool = dict(zip(seat_pool.player_id, seat_pool.table_id))
dev_features['table_id'] = dev_features.player_1.map(player_pool)
PAIR_FEATURES = [c for c in dev_features.columns
                 if c not in {'pair_id', 'player_1', 'player_2', 'label', 'behavior_family', 'table_id'}]
print('pair features:', len(PAIR_FEATURES), '| labelled pairs with rows:', len(dev_features))
dev_features.head(3)


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

def risk_model(seed):
    return Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                     ('model', ExtraTreesClassifier(n_estimators=300, min_samples_leaf=4, max_features=0.8,
                                                    class_weight='balanced', random_state=seed, n_jobs=-1))])

X = dev_features[PAIR_FEATURES].to_numpy(dtype=float)
y = dev_features.label.to_numpy(int)
groups = dev_features.table_id.to_numpy()
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))
dev_features['fold'] = -1
oof_risk = np.zeros(len(dev_features))
for k, (tr, va) in enumerate(folds):
    m = risk_model(SEED + k).fit(X[tr], y[tr])
    oof_risk[va] = m.predict_proba(X[va])[:, 1]
    dev_features.loc[dev_features.index[va], 'fold'] = k
dev_features['oof_risk'] = oof_risk
final_risk = risk_model(SEED).fit(X, y)

from sklearn.metrics import average_precision_score
print('OOF pair AP (trusted cohort):', round(average_precision_score(y, oof_risk), 6))
imp = pd.Series(final_risk['model'].feature_importances_[:len(PAIR_FEATURES)], index=PAIR_FEATURES)
print(imp.sort_values(ascending=False).head(12).round(4).to_string())


## 6. Behavior head

The metric scores each disclosed family one-vs-rest, using `risk_score` where the predicted label matches
and 0 otherwise. A pair labelled with the wrong family is therefore penalised twice, so the head is trained
only on confirmed targets and applied to every pair.


In [ ]:
positives = dev_features[dev_features.label.eq(1)].reset_index(drop=True)
Xp, yp = positives[PAIR_FEATURES].to_numpy(dtype=float), positives.behavior_family.to_numpy()
fam_oof = np.empty(len(positives), dtype=object)
for tr, va in GroupKFold(n_splits=N_FOLDS).split(Xp, yp, positives.table_id):
    fam = risk_model(SEED + 7).fit(Xp[tr], yp[tr])
    fam_oof[va] = fam.predict(Xp[va])
print('OOF family accuracy on confirmed targets:', round((fam_oof == yp).mean(), 4))
print(pd.crosstab(pd.Series(yp, name='true'), pd.Series(fam_oof, name='predicted')))
family_model = risk_model(SEED + 7).fit(Xp, yp)


## 7. Evidence ranker

Positive pairs list up to five hands with behaviour-specific evidence. An unlisted hand is *not* a
confirmed negative — the list is capped — so this is annotation retrieval and the ranker is trained on
listed-versus-other hands of the same positive pairs, with one model per family plus a generic fallback.


In [ ]:
import xgboost as xgb

HAND_FEATURES = [c for c in FEATURES]
listed = evidence.assign(listed=1)[['pair_id', 'hand_id', 'listed']]
pos_ids = set(positives.pair_id)
train_hands = dev_hand_rows[dev_hand_rows.pair_id.isin(pos_ids)].merge(listed, on=['pair_id', 'hand_id'], how='left')
train_hands['listed'] = train_hands.listed.fillna(0)
train_hands = train_hands.merge(dev_features[['pair_id', 'behavior_family', 'table_id', 'fold']], on='pair_id')
print('training hands:', len(train_hands), '| listed:', int(train_hands.listed.sum()))

def fit_ranker(frame, seed):
    frame = frame.sort_values('pair_id')
    groups = frame.groupby('pair_id', sort=False).size().to_numpy()
    d = xgb.DMatrix(frame[HAND_FEATURES].to_numpy(dtype=float), label=frame.listed.to_numpy(), group=groups,
                    missing=np.nan)
    params = dict(objective='rank:map', eta=0.1, max_depth=5, subsample=0.9, colsample_bytree=0.8,
                  min_child_weight=5, seed=seed, nthread=WORKERS, eval_metric='map@5')
    return xgb.train(params, d, num_boost_round=180)

def score_hands(model, frame):
    if not len(frame):
        return np.zeros(0)
    return model.predict(xgb.DMatrix(frame[HAND_FEATURES].to_numpy(dtype=float), missing=np.nan))

FAMILIES = sorted(positives.behavior_family.unique())
MIN_LISTED = 40          # below this a family ranker is unstable; fall back to the generic one

def family_rankers_for(frame, seed):
    """One ranker per family, but only where that family has enough listed hands to train on."""
    out = {}
    for f in FAMILIES:
        block = frame[frame.behavior_family.eq(f)]
        if block.listed.sum() >= MIN_LISTED:
            out[f] = fit_ranker(block, seed)
    return out

oof_scores = np.zeros(len(train_hands))
for k in range(N_FOLDS):
    tr, va = train_hands.fold != k, train_hands.fold == k
    generic = fit_ranker(train_hands[tr], SEED + 11 + k)
    per_family = family_rankers_for(train_hands[tr], SEED + 21 + k)
    block = train_hands[va]
    scores = score_hands(generic, block)
    for f, model in per_family.items():
        m = (block.behavior_family == f).to_numpy()
        if m.any():
            scores[m] = score_hands(model, block[m])
    oof_scores[va.to_numpy()] = scores
train_hands['oof_score'] = oof_scores

generic_model = fit_ranker(train_hands, SEED + 11)
family_rankers = family_rankers_for(train_hands, SEED + 21)
print('family rankers trained:', sorted(family_rankers) or '(none: generic fallback everywhere)')


## 8. The official metric, locally

`total = 0.70 * pair AP + 0.20 * evidence MAP@5 + 0.10 * behavior MAP`, with ties broken by `pair_id`.
Computed here on the trusted cohort out-of-fold, which is optimistic relative to the leaderboard: the
public set contains unknown positives and a fourth mechanism that has no labels here.


In [ ]:
def ranked_ap(y_true, scores, ids):
    order = np.lexsort((np.asarray(ids), -np.asarray(scores, dtype=float)))
    y = np.asarray(y_true)[order]
    if y.sum() == 0:
        return 0.0
    hits = np.cumsum(y)
    precision = hits / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())

def evidence_ap(relevant, predicted):
    truth, hit, total = set(relevant), 0, 0.0
    for rank, hand in enumerate(predicted[:5], 1):
        if hand in truth:
            hit += 1
            total += hit / rank
    return total / min(5, len(truth)) if truth else 0.0

top5 = (train_hands.sort_values(['pair_id', 'oof_score'], ascending=[True, False])
        .groupby('pair_id').hand_id.apply(lambda s: list(s.head(5))).to_dict())
ev_map = float(np.mean([evidence_ap(truth_evidence.get(p, []), top5.get(p, [])) for p in positives.pair_id]))
pair_ap = ranked_ap(dev_features.label, dev_features.oof_risk, dev_features.pair_id)
predicted_family = pd.Series(index=dev_features.pair_id, dtype=object)
predicted_family[:] = family_model.predict(dev_features[PAIR_FEATURES].to_numpy(dtype=float))
behavior = float(np.mean([
    ranked_ap((dev_features.behavior_family == f).astype(int),
              np.where(predicted_family.to_numpy() == f, dev_features.oof_risk, 0.0), dev_features.pair_id)
    for f in FAMILIES]))
print(f'pair AP        {pair_ap:.6f}')
print(f'evidence MAP@5 {ev_map:.6f}')
print(f'behavior MAP   {behavior:.6f}')
print(f'TOTAL (local)  {0.7*pair_ap + 0.2*ev_map + 0.1*behavior:.6f}')


## 9. Evaluation inference and submission

Each evaluation pool is processed on its own: build pair-hand rows, aggregate to pair features, score risk
and family, rank that pool's hands with the routed ranker, and keep only the top five per pair. Evidence
hands must be shared by both players and belong to the evaluation phase, which the final validation checks.


In [ ]:
wanted_eval = {}
for r in eval_pairs.itertuples(index=False):
    wanted_eval[pair_key(r.player_1, r.player_2)] = r.pair_id
eval_tasks = [(p, 'evaluation', wanted_eval) for p in pools]

risk_rows, evidence_rows = [], []
t0 = time.time()
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for i, part in enumerate(ex.map(pool_pair_hands, eval_tasks), 1):
        if not len(part):
            continue
        feats = aggregate_pairs(part)
        matrix = feats.reindex(columns=PAIR_FEATURES).to_numpy(dtype=float)
        feats['risk_score'] = final_risk.predict_proba(matrix)[:, 1]
        feats['predicted_behavior'] = family_model.predict(matrix)
        risk_rows.append(feats[['pair_id', 'risk_score', 'predicted_behavior']])
        block = part.merge(feats[['pair_id', 'predicted_behavior']], on='pair_id')
        scores = score_hands(generic_model, block)
        for f, model in family_rankers.items():
            m = (block.predicted_behavior == f).to_numpy()
            if m.any():
                scores[m] = score_hands(model, block[m])
        block['score'] = scores
        best = (block.sort_values(['pair_id', 'score'], ascending=[True, False])
                     .groupby('pair_id').head(5)[['pair_id', 'hand_id']])
        evidence_rows.append(best)
        if i % 50 == 0:
            print(f'evaluation {i}/{len(eval_tasks)} pools, {time.time()-t0:.0f}s', flush=True)

pair_scores = pd.concat(risk_rows, ignore_index=True)
evidence_top = pd.concat(evidence_rows, ignore_index=True)
print('scored pairs:', len(pair_scores), '| evidence rows:', len(evidence_top))


In [ ]:
wide = (evidence_top.assign(rank=evidence_top.groupby('pair_id').cumcount() + 1)
        .pivot(index='pair_id', columns='rank', values='hand_id')
        .reindex(columns=[1, 2, 3, 4, 5]))
wide.columns = [f'evidence_hand_{c}' for c in wide.columns]

submission = template[['pair_id']].merge(pair_scores, on='pair_id', how='left').merge(wide, on='pair_id', how='left')
submission['risk_score'] = submission.risk_score.fillna(0.0).clip(0, 1)
submission['predicted_behavior'] = submission.predicted_behavior.fillna('none')
for c in [f'evidence_hand_{i}' for i in range(1, 6)]:
    submission[c] = submission[c].fillna('NO_EVIDENCE')
submission = submission[template.columns.tolist()]

assert len(submission) == len(template), 'row count'
assert submission.pair_id.tolist() == template.pair_id.tolist(), 'row order must match the template'
assert submission.risk_score.between(0, 1).all() and np.isfinite(submission.risk_score).all()
assert submission.predicted_behavior.isin(
    {'none', 'directed_transfer', 'soft_play', 'coordinated_isolation', 'other_coordination'}).all()
ev_cols = [f'evidence_hand_{i}' for i in range(1, 6)]
dupes = submission[ev_cols].apply(lambda r: len({x for x in r if x != 'NO_EVIDENCE'}) !=
                                  len([x for x in r if x != 'NO_EVIDENCE']), axis=1)
assert not dupes.any(), 'repeated evidence within a pair is invalid'

# membership: every cited hand must be an evaluation hand shared by both endpoints
cited = pd.melt(submission, id_vars='pair_id', value_vars=ev_cols, value_name='hand_id')
cited = cited[cited.hand_id != 'NO_EVIDENCE'].merge(eval_pairs[['pair_id', 'player_1', 'player_2']], on='pair_id')
eval_hand_ids = set(hands[hands.phase.eq('evaluation')].hand_id)
assert set(cited.hand_id) <= eval_hand_ids, 'evidence must be evaluation-phase hands'
seat_check = pq.read_table(DATA_DIR / 'seats.parquet', columns=['hand_id', 'player_id']).to_pandas()
seat_check = seat_check[seat_check.hand_id.isin(set(cited.hand_id))]
seated = set(zip(seat_check.hand_id, seat_check.player_id))
assert all((h, a) in seated and (h, b) in seated
           for h, a, b in zip(cited.hand_id, cited.player_1, cited.player_2)), 'both players must be seated'

path = OUT_DIR / 'submission.csv'
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
print(submission.predicted_behavior.value_counts().to_dict())
submission.head()


## What this is, and what it is not

**Differences from the 0.82412 file.** That submission used a 700-field pair model (own-card and
price-conditioned response cells plus cross-fitted hidden-card residual moments), 930-field evidence
rankers with an added decision-equity block, family-routed evidence, and a multi-hour build with exact
replay. This notebook keeps the structure and a compact feature set, so it will score lower.

**Things worth knowing if you build on this.**

* The trusted-cohort pair AP saturates quickly (it is easy to separate 372 confirmed targets from 1,488
  confirmed non-targets). It is a weak guide to leaderboard movement. Scoring the full development
  population at a fixed budget is far more informative.
* Training on one family and testing on another collapses: features that express a family perfectly when it
  is supervised retrieve almost none of it when it is not. Plan for the undisclosed fourth mechanism
  accordingly — it cannot be learned from the three labelled families.
* Listed evidence hands cluster early in a pair's history **only** when exactly five hands are listed. In
  pairs with fewer than five the distribution is flat, so that skew is an artifact of capping a
  chronologically ordered list, not a behavioural signal. Using hand position as a feature exploits the
  label construction rather than gameplay.
* An unlisted hand of a positive pair is not a negative, and an unlabelled pair is not a non-target.
